In [3]:
import numpy as np
import data_handling
import read_file
import ploting
import glob

In [4]:
# import os
# work_path = os.getcwd().split("\\")[:-1]
# work_path = "\\".join(work_path)
# os.chdir(work_path)
# os.getcwd()

In [5]:
elev_path = r"Inputs\elev_bon.asc"
meta_elev, grid_elev = read_file.read_asc_file(file_path=elev_path,)
luse_path = r"Inputs\luse_bon.asc"
meta_luse, grid_luse = read_file.read_asc_file(file_path=luse_path,)

In [6]:
surface_path = r"Inputs\Surface_input.txt"
dict_soil = read_file.create_dict_luse(surface_path, )
mask_house = data_handling.create_mask_luse(dict_soil, grid_luse, ["House"])

In [7]:
grid_new_elev = grid_elev + 5*mask_house
np.nanmax(grid_new_elev)

np.float64(52.990928649902344)

In [8]:
header = "Titre \n"
meta = [f"{key} {value}\n" for key, value in meta_elev.items()]
header = header + "".join(meta)
grid_new_elev = np.nan_to_num(grid_new_elev, nan=-9999)
np.savetxt("elev_up.asc", grid_new_elev, fmt='%.18e', delimiter=' ', newline='\n', header=header, footer='', comments='', encoding=None)
  # for key, value in dict_soil.items():
  #   f.write(f"{key} {value}")
    
    
header = "Titre \n"
meta = [f"{key} {value}\n" for key, value in meta_elev.items()]
header = header + "".join(meta)
grid_elev_simp = grid_elev/grid_elev + 5*mask_house
grid_elev_simp = np.nan_to_num(grid_elev_simp, nan=-9999)
np.savetxt("elev_simp.asc", grid_elev_simp, fmt='%.18e', delimiter=' ', newline='\n', header=header, footer='', comments='', encoding=None)
  # for key, value in dict_soil.items():
  #   f.write(f"{key} {value}")


In [9]:
grid_elev_1 = grid_elev /grid_elev
header = "Titre \n"
meta = [f"{key} {value}\n" for key, value in meta_elev.items()]
header = header + "".join(meta)
grid_elev_1 = np.nan_to_num(grid_elev_1, nan=-9999)
np.savetxt("elev_0.asc", grid_elev_1, fmt='%.18e', delimiter=' ', newline='\n', header=header, footer='', comments='', encoding=None)
  # for key, value in dict_soil.items():
  #   f.write(f"{key} {value}")

In [10]:
grid_soil_1 = grid_luse/grid_luse
header = "Titre \n"
meta = [f"{key} {value}\n" for key, value in meta_elev.items()]
header = header + "".join(meta)
grid_soil_1 = np.nan_to_num(grid_soil_1, nan=-9999)

np.savetxt("luse_1.asc", grid_soil_1, fmt='%.0f', delimiter=' ', newline='\n', header=header, footer='', comments='', encoding=None)
np.savetxt("soils_1.asc", grid_soil_1, fmt='%.0f', delimiter=' ', newline='\n', header=header, footer='', comments='', encoding=None)
  # for key, value in dict_soil.items():

In [11]:
import plotly.graph_objects as go
#### Création des autres cartes 
def create_plotly_map(grid: np.array, metadata: dict, 
                      title: str = "Titre", color: str = 'blues_r', fig_dim: tuple = (None, None), 
                      unit: list = ["Data", "unit"], 
                      grids_hover: np.array = None, info_hover: list =None) -> go.Figure:
    """Create an interactive heatmap with continuous colorbar using Plotly, based on a 2D grid and spatial metadata.
    The x and y axes represent geographic coordinates, while the hover information includes pixel indices
    Additional grids can be provided to enrich hover information (e.g., land use, interception depth).
    
    Args:
        grid (np.ndarray):
            2D array representing the data to be visualized.
        metadata (dict):
            dict_soilnary containing spatial metadata for the grid. Must include:
            - 'xllcorner': x-coordinate of the lower-left corner.
            - 'yllcorner': y-coordinate of the lower-left corner.
            - 'cellsize': Size of each grid cell.
            - 'ncols': Number of columns in the grid.
            - 'nrows': Number of rows in the grid.
        title (str, optional):
            Title of the plot. Defaults to "Titre".
        color (str, optional):
            Color scale for the heatmap. Defaults to 'blues_r'.
        fig_dim (tuple, optional):
            Figure dimensions as (width, height). Defaults to (None, None).
        unit (list, optional):
            List containing the data label and unit (e.g., ["Elevation", "m"]).
            Defaults to ["Data", "unit"].
        grids_hover (list of np.ndarray, optional):
            List of additional 2D grids to include in hover information.
            Each grid must match the shape of `grid`. Defaults to None.
        info_hover (list of str, optional):
            List of labels for the additional hover information.
            Must match the number of grids in `grids_hover`. Defaults to None.
    
    Notes:
        - Pixel indices in the hover information start at 1 (to match TREX conventions).
        - If `grids_hover` is provided, `info_hover` must have the same number of elements.
        - The y-axis is reversed to match cartographic conventions (y increases upwards).
        - Not suited for ploting the land use map.

    Returns:
        go.Figure: A Plotly Figure object containing the interactive heatmap.
    """
    # Coordination spatiale
    x0 = metadata['xllcorner']
    y0 = metadata['yllcorner']
    dx = metadata['cellsize']
    nx = metadata['ncols']
    ny = metadata['nrows']
    x = np.linspace(x0, x0 + nx * dx, nx)
    y = np.linspace(y0, y0 + ny * dx, ny)
    # Inversion des valeurs de y 
    y = y[::-1]
    # Valeurs extrèmes pour la colorbar
    maxi = np.nanmax(grid)
    mini = np.nanmin(grid)
    # Pour le hover indices des pixels
    rows, cols = np.indices(grid.shape)
    customdata_test = np.dstack((rows +1, cols +1)) # adding 1 to match the way TREX count (starting to 1)
    # Adding the hovers grids to customdata
    if grids_hover:
        for grid_h in grids_hover:
            customdata_test = np.dstack((customdata_test, grid_h))
    # Création de la fig
    fig = go.Figure()
    fig.add_trace(
        go.Heatmap(
            z=grid,
            zmax=maxi, zmin=mini,
            x=x, y=y,
            customdata=customdata_test,
            hovertemplate=f"<b>{unit[0]}:" +"%{z:.10f}" + f" {unit[1]}<extra></extra></b><br>"+
                            "Ligne: %{customdata[0]}<br>" +
                            "Colonne: %{customdata[1]}<br>",
            colorscale=color
        )
    )
    # Ajout des informations supplémentaires pour chaque grille dans grids_hover
    if grids_hover and info_hover:
         # Récupération du hovertemplate de base
        base_hovertemplate = fig.data[0].hovertemplate
        for i, info in enumerate(info_hover, start=2):
            base_hovertemplate += f"{info}: %"+ "{customdata[" + f"{i}]" + "}<br>" # Weird syntax to match plotly needs
        # Mise à jour du hovertemplate
        fig.data[0].hovertemplate = base_hovertemplate
    fig.update_layout(
        title=title,
        autosize = False,
        width=fig_dim[0],
        height=fig_dim[1],
        xaxis_title="X (m)",
        xaxis=dict(
        tickformat=" d",  # Affiche les entiers 
        tickmode='auto',
        showgrid=True
        ),
        yaxis_title="Y (m)",
        yaxis=dict(
        tickformat=" d",  # Affiche les entiers
        tickmode='auto',
        showgrid=True
        ),
        )
    fig.update_yaxes(scaleanchor="x")
    fig.update_coloraxes(colorbar=dict(nticks=10, ticks="outside",ticklen=6, ))
    
    return fig


meta, grid_lidar = read_file.read_asc_file(r"Inputs\elev_bon.asc")
map_lidar = create_plotly_map(grid_lidar, meta, title="Elev LIiDAR", fig_dim=(1000,1000))
map_lidar.show()

In [21]:
glob.glob("/Inputs*.html")

[]

In [29]:
import os
path = "Outputs/Grids/Rain/raindepth"

path2 = "/".join(path.split("/")[:-1])
print(path2)
os.listdir("Outputs/Grids/Rain")

Outputs/Grids/Rain


['raindepth.0',
 'raindepth.1',
 'raindepth.10',
 'raindepth.11',
 'raindepth.12',
 'raindepth.13',
 'raindepth.14',
 'raindepth.15',
 'raindepth.16',
 'raindepth.17',
 'raindepth.18',
 'raindepth.19',
 'raindepth.2',
 'raindepth.20',
 'raindepth.21',
 'raindepth.22',
 'raindepth.23',
 'raindepth.24',
 'raindepth.25',
 'raindepth.26',
 'raindepth.27',
 'raindepth.28',
 'raindepth.29',
 'raindepth.3',
 'raindepth.30',
 'raindepth.31',
 'raindepth.32',
 'raindepth.33',
 'raindepth.34',
 'raindepth.35',
 'raindepth.36',
 'raindepth.37',
 'raindepth.38',
 'raindepth.39',
 'raindepth.4',
 'raindepth.40',
 'raindepth.5',
 'raindepth.6',
 'raindepth.7',
 'raindepth.8',
 'raindepth.9',
 'rainrate.0',
 'rainrate.1',
 'rainrate.10',
 'rainrate.11',
 'rainrate.12',
 'rainrate.13',
 'rainrate.14',
 'rainrate.15',
 'rainrate.16',
 'rainrate.17',
 'rainrate.18',
 'rainrate.19',
 'rainrate.2',
 'rainrate.20',
 'rainrate.21',
 'rainrate.22',
 'rainrate.23',
 'rainrate.24',
 'rainrate.25',
 'rainrate.2